# TODO 6.4：SmoothQuant INT8 / FP8 能力验证

使用 NVIDIA ModelOpt 对真实模型执行 W8A8 SmoothQuant。FP8 只有在目标设备成功完成真实 FP8 矩阵运算后才记为 `pass`；仅存在 PyTorch FP8 dtype 不算通过。完整 INT8 状态写入 `banking_llm_project`，分析证据写入 `OptimizationSystem/out`。

In [ ]:
%pip -q install -U "nvidia-modelopt[torch]" accelerate safetensors
%pip -q install -U transformers==5.18.0
print('安装完成。请重启 Colab 会话；重启后跳过本单元。')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
CONFIG_PATH = PROJECT_DIR / 'configs/int8_fp8.json'
RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
MODEL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/int8_fp8')
sys.path.insert(0, str(PROJECT_DIR))

from src.data.finqa_assets import load_assets
from src.quantization.int8_fp8 import (
    build_smoothquant_plan, collect_int8_fp8_evidence,
    run_smoothquant_plan, validate_int8_fp8_evidence,
)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU 运行时。'
assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plan = build_smoothquant_plan(config)
print('GPU:', torch.cuda.get_device_name(0))
print('校准样本:', len(assets.calibration_prompts))
print('计划:', plan.to_dict())

In [ ]:
plan_result_dir = RESULT_DIR / plan.plan_id
plan_model_dir = MODEL_ROOT / plan.plan_id
manifest = plan_result_dir / 'quantized_model_manifest.json'
if manifest.is_file():
    print('跳过已完成导出:', manifest)
else:
    result = run_smoothquant_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=plan_result_dir,
        model_output_dir=plan_model_dir,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
        calibration_batch_size=int(config['calibration_batch_size']),
    )
    if result['status'] != 'pass':
        raise RuntimeError(result.get('detail', result))
    gc.collect(); torch.cuda.empty_cache()
print('INT8 manifest:', manifest)

In [ ]:
evidence = collect_int8_fp8_evidence(CONFIG_PATH, CALIBRATION_MANIFEST, RESULT_DIR)
evidence_path = RESULT_DIR / 'int8_fp8_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_int8_fp8_evidence(evidence_path)
payload = report.to_dict()
(RESULT_DIR / 'validation.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, ensure_ascii=False, indent=2))
statuses = {item['name']: item['status'] for item in payload['checks']}
assert statuses['smoothquant_real_execution'] == 'pass'
print('SmoothQuant INT8 真实导出：PASS')
print('NVIDIA FP8:', statuses['nvidia_fp8_capability'])
print('Ascend FP8:', statuses['ascend_fp8_capability'])